# 02. Проверка архитектур
Шесть основных моделей и Hybrid-AE/Transformer-VAE для абляций. Проверяются реальные AE/VAE loss и градиенты всех параметров.


In [ ]:
from pathlib import Path
import sys
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "src" / "tsprep").is_dir() and (p / "pyproject.toml").exists()), None)
if ROOT is None:
    raise RuntimeError("Run this notebook inside the tsprep project")
sys.path.insert(0, str(ROOT / "src"))
import numpy as np
import matplotlib.pyplot as plt
import torch
torch.set_num_threads(2)


In [ ]:
from tsprep.config import ALL_MODEL_NAMES, ABLATION_MODEL_NAMES, ModelConfig
from tsprep.models.autoencoder import build_model
from tsprep.mlcore.losses import compute_loss
torch.manual_seed(42)
x = torch.randn(8, 48, 1)
for name in ALL_MODEL_NAMES + ABLATION_MODEL_NAMES:
    cfg = ModelConfig(model_name=name, hidden_dim=32, d_model=32, tcn_levels=3)
    model = build_model(cfg, 1, 48).train()
    x_hat, z, mu, logvar = model(x)
    loss, parts = compute_loss(x_hat, x, mu, logvar, model.variational, "mse", .01)
    loss.backward()
    assert x_hat.shape == x.shape and torch.isfinite(loss)
    assert all(p.grad is not None and torch.isfinite(p.grad).all() for p in model.parameters() if p.requires_grad)
    print(name, "parameters:", sum(p.numel() for p in model.parameters()), "loss:", parts)


Регрессионные проверки: `python -m unittest discover -s tests -p test_regressions.py -v`.


## Проверка двух целей реконструкции
Используется один и тот же noisy input и небольшой детерминированный TCN-AE. Меняется только target: observed либо clean. Это проверка loss/градиентов, а не оценка качества denoising. Отдельный сравнительный запуск находится в ноутбуке 05.

In [ ]:
from tsprep.data_preprocessing.synthetic import generate_series
from tsprep.data_preprocessing.datasets import prepare_and_window, build_dataloaders
from tsprep.config import DataConfig
sample = generate_series(length=700, seed=9001)
data_cfg = DataConfig(window_len=24, stride=2, batch_size=64)
observed = prepare_and_window(sample.values, data_cfg)
clean = prepare_and_window(sample.clean_values, data_cfg, normalizer=observed.normalizer)
paired, _, _ = build_dataloaders(observed, data_cfg, train_targets=clean.train_windows)
noisy_x, clean_x = next(iter(paired))
small_model = build_model(ModelConfig(model_name="tcn_ae", hidden_dim=16, latent_dim=8, tcn_levels=2, dropout=0), 1, 24)
prediction, _, mu, logvar = small_model(noisy_x)
ordinary, _ = compute_loss(prediction, noisy_x, mu, logvar, False, "mse", 0.)
denoising, _ = compute_loss(prediction, clean_x, mu, logvar, False, "mse", 0.)
denoising.backward()
assert torch.isfinite(denoising) and all(p.grad is not None for p in small_model.parameters())
print({"observed_target_mse": ordinary.item(), "clean_target_mse": denoising.item(), "shape": tuple(prediction.shape)})